# James Merrill — Workshop Notebook
**CompLit 126x — Love in Context**

This notebook implements the prompt chain designed by the James Merrill workshop group. Their chain used a **batch poems + self-critique loop (×3)** architecture — feeding in existing Merrill poems, applying specific thematic critiques, then running the model through three rounds of self-evaluation and revision.

```
one-shot  ──▶  give existing  ──▶  critiques of
sonnet         poems (batch)       theme / content
                                        │
                    ┌───────────────────┘
                    ▼
              self-critique ──▶ revise ─┐
                    ▲                   │
                    └───── ×3 ──────────┘
                                        │
                                        ▼
                                   final poem ✦
```

**What makes this chain interesting:** The group originally used Claude (noted in red on their diagram), but the architecture works with any model. The self-critique loop is the key move: instead of *you* telling the model what's wrong, you ask the model to critique *itself*, then improve based on its own assessment — and repeat three times. Each pass catches different problems, because the critique is always responding to the latest version.

---
**Run the cells in order.** Each step builds on the previous one.

## Setup
Run the two cells below once at the start of your session.

In [ ]:
# Install the OpenAI SDK (run once per session)
%pip install openai --quiet
print("✓ Installed")

In [ ]:
from openai import OpenAI
import json
import os

# ── API Key ──────────────────────────────────────────────────────────────────
# In Google Colab:
#   1. Click the 🔑 (Secrets) icon in the left sidebar
#   2. Add a secret named  OPENAI_API_KEY  with your key
#   3. Toggle "Notebook access" to ON, then run this cell
#
# Locally: set the OPENAI_API_KEY environment variable

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    print("✓ Using Colab Secrets")
except (ImportError, Exception):
    api_key = os.environ.get('OPENAI_API_KEY')
    print("✓ Using environment variable")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o"

# Helper: call the model and return the text
def ask(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.responses.create(model=MODEL, input=messages)
    return response.output_text

print(f"✓ Client ready | Model: {MODEL}")

---
## Step 1: One-Shot (Baseline)

Start with a simple prompt. The group found the one-shot was "alright" — it captured some Merrill surface features but missed the deeper mechanics.

In [ ]:
# ── Step 1: One-shot ─────────────────────────────────────────────────────────

one_shot = ask(
    "Generate a sonnet in the style of James Merrill."
)

print("ONE-SHOT RESULT")
print("═" * 60)
print(one_shot)
print("\n" + "═" * 60)
print("\n↓ The group found this was 'alright' — Merrill-ish on the")
print("  surface, but missing his particular wit, formal play,")
print("  and the way he braids the mundane with the metaphysical.")

---
## Step 2: Feed In Existing Poems

The group gave the model a batch of five existing Merrill poems to study. This shifts the model from relying on its general training data to working from specific examples.

**Paste Merrill poems below.** The more specific the source material, the better the output. These selections cover a range of Merrill's modes: the domestic-metaphysical, the witty-elegiac, the formally dazzling.

In [ ]:
# ── Step 2: Feed in existing poems ───────────────────────────────────────────
# Paste actual Merrill poems here. Replace or add more as needed.

MERRILL_POEMS = [
    {
        "title": "A Timepiece",
        "text": """Of a pendulum's mildness, with her feet up
My            aunt            lay            expecting            her            third            child.
Over the racked crib the record-Loss, come to terms
With the clock, the plow, the rocking chair.
Not to be counted on to change the subject."""
    },
    {
        "title": "The Victor Dog",
        "text": """Bix to Buxtehude to Boulez,
The little white dog on the Victor label
Listens long and hard as he is able.
It's all in a day's work, whatever plays.

From judgment, it would seem, he has refrained.
He even likes Stravinsky, he## goes to show
There are more things in heaven and earth than are dreamed of
In our philosophy. He has not yet learned to grow old."""
    },
    {
        "title": "An Urban Convalescence",
        "text": """Out for a walk, after a week in bed,
I find them tearing up part of my block
And, chilled through, dazed and lonely, join the dozen
In meek attitudes, watching a huge crane
Fumble luxuriously in the filth of years.
Her jaws dribble rubble. An old man
Laughs and curses in her survey."""
    },
    {
        "title": "The Broken Home",
        "text": """Crossing the street,
I saw the parents and the child at their door.
She was survey each of my survey,
The survey survey survey to the survey survey.
The survey survey survey to the survey.
Father survey survey survey his survey.
A survey survey survey survey survey."""
    },
    {
        "title": "b o d y",
        "text": """Look closely at the letters. Can you see,
entering (deep into the survey of survey),
how the survey survey were survey to survey?"""
    }
]

poems_text = "\n\n---\n\n".join(
    f"\"{p['title']}\":\n{p['text']}" for p in MERRILL_POEMS
)

# Generate improved poem with the batch
batch_poem = ask(
    f"""Here are several poems by James Merrill for you to study:

{poems_text}

Now write a new sonnet in Merrill's style. Pay close attention to:
- His characteristic wit and wordplay
- The way he moves between the domestic and the metaphysical
- His formal dexterity — puns, enjambment, tonal shifts
- The elegance of his syntax (long, winding sentences that reward
  re-reading)
- His ability to make the mundane shimmer with double meaning

Do not copy his lines — write something new that belongs in his world.

Write only the poem."""
)

print(f"✓ Fed in {len(MERRILL_POEMS)} Merrill poems")
print("\nBATCH-INFORMED POEM")
print("═" * 60)
print(batch_poem)

---
## Step 3: Critiques of Theme and Content

The group had specific critiques about what the batch-informed poem still got wrong — not just surface style but deeper issues of **theme and content**. This step applies those targeted critiques and asks for a revision.

**Edit the critiques below** to match what you notice about the poem above. What themes are missing or mishandled? What content feels generic rather than specifically Merrill?

In [ ]:
# ── Step 3: Critiques of theme and content ───────────────────────────────────
# Edit these critiques based on what you see in the batch poem above.

CRITIQUES = """
Specific critiques of the poem above:

1. THEME — Merrill's poems are rarely *about* one thing. They layer
   multiple subjects (a broken teacup AND a broken marriage AND the
   nature of form itself). The poem above is too single-minded.

2. CONTENT — Merrill draws from specific, often autobiographical
   material: Stonington, Greece, his parents' divorce, the Ouija board,
   opera, domestic objects with histories. The poem above uses generic
   "poetic" content instead of Merrill's particular world.

3. WIT — Merrill's humor is built into the language itself: puns,
   double meanings, words that split open. The poem above may gesture
   at cleverness but doesn't embed wit in the diction.

4. TONAL RANGE — Merrill can be camp and devastating in the same line.
   The poem above likely holds a single tone throughout.
"""

critiqued_poem = ask(
    f"""Here is a sonnet written in the style of James Merrill:

{batch_poem}

Here are specific critiques of its theme and content:

{CRITIQUES}

Revise the poem to address these critiques. Keep what works.
Fix what doesn't. The revised poem should feel like it belongs
in Merrill's world — not just his style, but his *subject matter*
and his way of thinking.

Write only the revised poem."""
)

print("CRITIQUED + REVISED POEM")
print("═" * 60)
print(critiqued_poem)

---
## Step 4: Self-Critique Loop (×3)

This is the group's signature move. Instead of *you* critiquing the poem again, you ask the **model to critique itself** — then revise based on its own assessment — and repeat **three times**.

Each pass through the loop catches different issues:
- **Round 1** usually fixes the most obvious problems (clichés, tonal mismatch)
- **Round 2** catches subtler issues (rhythm, the gap between intention and execution)
- **Round 3** refines (word-level choices, the final polish)

The cell below runs all three rounds automatically and shows you each version.

In [ ]:
# ── Step 4: Self-critique loop ×3 ────────────────────────────────────────────

current_poem = critiqued_poem
versions = []  # Store each round's output

for i in range(3):
    round_num = i + 1
    print(f"\n{'─' * 60}")
    print(f"ROUND {round_num} OF 3")
    print(f"{'─' * 60}")

    # Step A: Self-critique
    critique = ask(
        f"""Here is a sonnet written in the style of James Merrill:

{current_poem}

And here are actual Merrill poems for comparison:

{poems_text}

Critique this poem honestly. Compare it to the actual Merrill poems.
Be specific:
1. Which lines work well and why?
2. Which lines fall short of Merrill's standard?
3. Where is the wit genuine vs. forced?
4. Does the poem layer multiple subjects the way Merrill does?
5. What specific revisions would bring it closer to Merrill's voice?

Quote specific lines in your critique.""",
        system="You are a poetry critic and scholar of James Merrill's work. Be honest and specific."
    )

    print(f"\nSELF-CRITIQUE (round {round_num}):")
    print(critique[:500] + "..." if len(critique) > 500 else critique)

    # Step B: Revise based on self-critique
    revised = ask(
        f"""Here is a sonnet in the style of James Merrill:

{current_poem}

Here is a critique of this poem:

{critique}

Revise the poem to address the critique. Keep what the critic
praised. Fix what they identified as weak. Stay in Merrill's voice.

Write only the revised poem."""
    )

    print(f"\nREVISED POEM (round {round_num}):")
    print(revised)

    versions.append({
        "round": round_num,
        "critique": critique,
        "poem": revised
    })
    current_poem = revised

final_poem = current_poem
print("\n" + "═" * 60)
print("✓ Self-critique loop complete (3 rounds)")

---
## Compare All Versions

Now look at the full progression: from the one-shot through the batch-informed poem, the thematic critique, and each round of the self-critique loop.

In [ ]:
# ── Side-by-side comparison ──────────────────────────────────────────────────

print("PROGRESSION")
print("\n" + "═" * 60)
print("1. ONE-SHOT (no context)")
print("═" * 60)
print(one_shot)

print("\n" + "═" * 60)
print("2. BATCH-INFORMED (after feeding existing poems)")
print("═" * 60)
print(batch_poem)

print("\n" + "═" * 60)
print("3. AFTER THEME/CONTENT CRITIQUE")
print("═" * 60)
print(critiqued_poem)

for v in versions:
    print("\n" + "═" * 60)
    print(f"4.{v['round']}. SELF-CRITIQUE ROUND {v['round']}")
    print("═" * 60)
    print(v["poem"])

print("\n" + "═" * 60)
print("\nFor your essay, consider:")
print("  → What did the batch of existing poems add?")
print("  → What did your theme/content critiques catch?")
print("  → Across the 3 self-critique rounds, did quality improve")
print("    steadily, or plateau? Which round mattered most?")
print("  → What can the model critique about itself — and what can't it see?")
print("  → Is self-critique a genuine improvement or just polishing?")

---
## Going Further

This chain is a starting point. Here are ways to extend it for your assignment:

**Add more poems to the batch.** The group used five. Try ten. Does more source material make the self-critique more accurate, or does the model just have more surface features to copy?

**Run more self-critique rounds.** Try 5 or 7 rounds instead of 3. Does the poem keep improving, or does it start to flatten out? Finding the plateau is interesting data for your essay.

**Vary the critic persona.** The self-critique uses a "poetry critic and Merrill scholar" system prompt. What if the critic were a rival poet? A student? A hostile reviewer? Does the persona change what gets caught?

**Cross-chat the final poem.** Take the final poem into a fresh context (no history of the chain) and ask for a blind critique. Does a fresh model see problems the self-critique loop missed?

**Generate love song lyrics.** Swap the generation prompt to ask for song lyrics. Merrill's wit and wordplay might translate unusually well — or unusually badly — to verse-chorus-bridge form.

**Submitting your work:**
- **Lyrics**: Submit an album's worth of songs, with your favorite first
- **Audio**: Take your best lyrics to [Suno](https://suno.com) and generate audio
- **Essay** (500–700 words): Explain your prompt chain, include sample prompts, and reflect on what GPT-4o got right and wrong about your poet